# Primero importar 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

exp_name = "invarianeditexp2_cinematica"


archivo_completo = (
    f"kinematics/{exp_name}_completa.txt"
)

df = pd.read_csv(archivo_completo, sep='\t')

# Columnas disponibles: time | x_relativa | leg_x | body_x | leg_x_smooth | body_x_smooth
tiempo_s   = df['time'].values          # segundos
x_relativa = df['x_relativa'].values    # posición relativa pata−cuerpo (filtrada)
leg_x      = df['leg_x_smooth'].values  # pata X suavizada
body_x     = df['body_x_smooth'].values # cuerpo X suavizado


pata_x     = x_relativa
tiempo_pata_ms = tiempo_s * 1000.0     

print(f"Columnas: {list(df.columns)}")
print(f"Frames cargados : {len(df)}")
print(f"Duración total  : {tiempo_s[-1]:.2f} s")
df.head()

In [ ]:
# ── 1. DEFINE TU REGIÓN DE INTERÉS (En segundos) ──
tiempo_inicio = 3.0  
tiempo_fin = 58.0    


# Creamos una "máscara" que se queda solo con los datos dentro de ese intervalo
mascara_roi = (tiempo_s >= tiempo_inicio) & (tiempo_s <= tiempo_fin)

t_roi = tiempo_s[mascara_roi]
x_roi_recortado = x_relativa[mascara_roi]

print(f"Datos recortados: Analizando desde {tiempo_inicio}s hasta {tiempo_fin}s.")


plt.figure(figsize=(15, 5))
plt.plot(t_roi, x_roi_recortado, color='green', linewidth=1.5, label='Señal Recortada (ROI)')

plt.title(f'Región de Interés (ROI) Seleccionada', fontsize=16)
plt.xlabel('Tiempo (Segundos)', fontsize=12)
plt.ylabel('Posición Relativa X (Píxeles)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
from scipy.signal import find_peaks
from sklearn.linear_model import LinearRegression

x_roi_filtrada = x_roi_recortado


distancia_minima_frames = 20

picos_max, _ = find_peaks(x_roi_filtrada, distance=distancia_minima_frames)
picos_min, _ = find_peaks(-x_roi_filtrada, distance=distancia_minima_frames)

periodos = []
amplitudes = []

for i in range(len(picos_max) - 1):
    idx_actual = picos_max[i]
    idx_siguiente = picos_max[i+1]

    T = t_roi[idx_siguiente] - t_roi[idx_actual]

    segmento_onda = x_roi_filtrada[idx_actual:idx_siguiente]
    valle_real = np.min(segmento_onda)
    A = x_roi_filtrada[idx_actual] - valle_real

    periodos.append(T)
    amplitudes.append(A)

print(f"Pasos completos detectados en la ROI: {len(periodos)}")

In [ ]:
# Configurar el tamaño del gráfico para que se vea bien el detalle
plt.figure(figsize=(15, 6))



plt.plot(t_roi, x_roi_filtrada, color='blue', linewidth=2, label='Señal')

# 3. Marcamos los picos máximos detectados con puntos ROJOS
plt.plot(t_roi[picos_max], x_roi_filtrada[picos_max], "ro", markersize=8, label='Picos Máximos')

# 4. Marcamos los picos mínimos (valles) detectados con puntos VERDES
plt.plot(t_roi[picos_min], x_roi_filtrada[picos_min], "go", markersize=8, label='Valles Mínimos')



plt.xlabel('Tiempo (Segundos)', fontsize=14)
plt.ylabel('Posición Relativa X (Píxeles)', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)

# Movemos la leyenda fuera del gráfico para que no tape las ondas
plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))

# Ajustar los márgenes y mostrar
plt.tight_layout()
plt.show()

# Optimización para encontrar mejor R^2

# Analisis de velocidad y estabilidad:


In [ ]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import find_peaks, savgol_filter
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from pathlib import Path


plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         12,
    'axes.linewidth':    1.2,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'xtick.major.size':  4,
    'ytick.major.size':  4,
    'figure.dpi':        150,
    'savefig.dpi':       300,
})

COLOR_MEDIA     = '#2166ac'
COLOR_PICO_PICO = '#1a7c3e'
COLOR_SEÑAL     = '#888888'
COLOR_ROI       = '#2166ac'
COLOR_PICO      = '#d6604d'   # rojo para máximos
COLOR_MINIMO    = '#4dac26'   # verde para mínimos

def graficar_cuadrado(m, umbral, output_dir, exp_name, tiempo_s, x_relativa):
    X_gan = m['X']
    y_m   = m['y_media']
    y_pp  = m['y_pp']
    mod_m = m['modelo_m']
    mod_p = m['modelo_p']

    periodos_flat = X_gan.flatten()
    n = len(periodos_flat)
    
    # --- EXTRACCIÓN DE ESTADÍSTICAS PARA EL TFG ---
    media_periodo = np.mean(periodos_flat)
    std_periodo   = np.std(periodos_flat)
    media_amp     = np.mean(y_m)
    std_amp       = np.std(y_m)
    r2_m          = mod_m.score(X_gan, y_m)
    
    print(f"\n  ▶ RESULTADOS FÍSICOS: {exp_name} (Umbral {umbral})")
    print(f"    - Nº de pasos analizados: {n}")
    print(f"    - Periodo de oscilación:  {media_periodo:.3f} ± {std_periodo:.3f} s")
    print(f"    - Amplitud media (norm):  {media_amp:.3f} ± {std_amp:.3f}")
    print(f"    - R² (Amplitud vs Periodo): {r2_m:.4f}")
    print("-" * 50)

    # Variables para gráfica
    t_ini_g  = m['t_inicio']
    t_fin_g  = m['t_fin']
    mask_roi = (tiempo_s >= t_ini_g) & (tiempo_s <= t_fin_g)
    x_roi_g  = x_relativa[mask_roi]
    t_roi_g  = tiempo_s[mask_roi]
    x_filt_g = savgol_filter(x_roi_g, window_length=m['window_len'], polyorder=3)

    # Máximos (picos)
    p_max_g, _ = find_peaks(x_filt_g, distance=m['distancia'])

    # Mínimos: índices guardados en el dict
    p_min_g = np.array(m['minimos'], dtype=int)

    fig = plt.figure(figsize=(12, 14))
    gs  = gridspec.GridSpec(3, 1, height_ratios=[1, 2, 2], hspace=0.5)
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1])
    ax2 = fig.add_subplot(gs[2])


    ax0.plot(tiempo_s, x_relativa,
             color=COLOR_SEÑAL, linewidth=0.6, alpha=0.5, label='Señal completa')
    ax0.plot(t_roi_g, x_roi_g,
             color=COLOR_ROI, linewidth=1.0,
             label=f'ROI ({t_ini_g:.1f}–{t_fin_g:.1f} s)')
    ax0.plot(t_roi_g[p_max_g], x_roi_g[p_max_g],
             'v', color=COLOR_PICO, markersize=5, alpha=0.8,
             label=f'Picos / máximos ({len(p_max_g)})')

    # Mínimos — solo si hay alguno dentro del ROI
    if len(p_min_g) > 0:
        ax0.plot(t_roi_g[p_min_g], x_roi_g[p_min_g],
                 '^', color=COLOR_MINIMO, markersize=5, alpha=0.8,
                 label=f'Valles / mínimos ({len(p_min_g)})')

    ax0.axvspan(t_ini_g, t_fin_g, alpha=0.07, color=COLOR_ROI)
    ax0.set_xlabel('Tiempo (s)', fontsize=11)
    ax0.set_ylabel('x relativa (px)', fontsize=11)
    ax0.set_title(f'Señal completa — ROI ({exp_name})', fontsize=12, fontweight='bold')
    ax0.legend(fontsize=9, bbox_to_anchor=(1.02, 1), loc='upper left', framealpha=0.8)


    ax1.scatter(periodos_flat, y_m,
                color=COLOR_MEDIA, s=70, alpha=0.75,
                edgecolors='white', linewidths=0.5,
                label=f'Pasos (n = {n})')

    r2_media = mod_m.score(X_gan, y_m)
    if r2_media > 0.5:
        x_line = np.linspace(periodos_flat.min(), periodos_flat.max(), 200).reshape(-1, 1)
        y_line = mod_m.predict(x_line)
        ax1.plot(x_line, y_line,
                 linestyle='--', color='red', linewidth=2,
                 label=f'Regresión (R² = {r2_media:.3f})')

    ax1.set_title('Período vs Amplitud media', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Amplitud media (Norm. 0-1)', fontsize=11)
    ax1.set_xlabel('Período (s)', fontsize=11)
    ax1.legend(fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left', framealpha=0.85)
    ax1.set_box_aspect(1)


    ax2.scatter(periodos_flat, y_pp,
                color=COLOR_PICO_PICO, s=70, alpha=0.75,
                edgecolors='white', linewidths=0.5,
                label=f'Pasos (n = {n})')

    r2_pp = mod_p.score(X_gan, y_pp)
    if r2_pp > 0.5:
        x_line = np.linspace(periodos_flat.min(), periodos_flat.max(), 200).reshape(-1, 1)
        y_line = mod_p.predict(x_line)
        ax2.plot(x_line, y_line,
                 linestyle='--', color='red', linewidth=2,
                 label=f'Regresión (R² = {r2_pp:.3f})')

    ax2.set_title('Período vs Amplitud Pico a Pico', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Amplitud pico a pico (Norm. 0-1)', fontsize=11)
    ax2.set_xlabel('Período (s)', fontsize=11)
    ax2.legend(fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left', framealpha=0.85)
    ax2.set_box_aspect(1)

    fig.suptitle(
        f'MIN_PASOS = {umbral}  |  w_len: {m["window_len"]}  |  dist: {m["distancia"]}  |  R2 = {r2_m:.3f}',
        fontsize=11, color='#444444', y=0.98
    )

    plt.tight_layout()

    out_pdf = output_dir / f"{exp_name}_MIN{umbral}.pdf"
    out_svg = output_dir / f"{exp_name}_MIN{umbral}.svg"
    plt.savefig(out_pdf, format='pdf', bbox_inches='tight')
    plt.savefig(out_svg, format='svg', bbox_inches='tight')
    plt.close()


#incluir el nombre de todos los experimentos que quieras analizar (deben tener su archivo .txt correspondiente en la carpeta "kinematics/")
lista_experimentos = [
    "invarianeditexp2_cinematica",
    "varianteditexp2_cinematica",
]

base_path = "kinematics/"

for exp_name in lista_experimentos:
    print(f"\n=======================================================")
    print(f" INICIANDO ANÁLISIS: {exp_name}")
    print(f"=======================================================")

    archivo_completo = f"{base_path}{exp_name}_completa.txt"

    try:
        df = pd.read_csv(archivo_completo, sep='\t')
    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo {archivo_completo}. Saltando al siguiente...")
        continue

    tiempo_s   = df['time'].values
    x_relativa = df['x_relativa'].values

    print(f"Frames cargados : {len(df)} | Duración total: {tiempo_s[-1]:.2f} s")

    # Parámetros de búsqueda
    tiempo_fin          = tiempo_s[-1]
    inicios_posibles    = np.arange(2.0, 30.0, 0.5)
    fines_posibles      = np.arange(45.0, tiempo_fin, 0.5)
    windows_posibles    = [8, 10, 11, 12, 13, 15, 16, 17, 18, 19]
    distancias_posibles = [10, 11, 12, 13, 14, 15, 18, 22, 25]

    UMBRALES         = [55, 50, 45, 40]
    MIN_PASOS_GLOBAL = min(UMBRALES)

    resultados = []

    for t_ini, t_fin, w_len, d_min in itertools.product(
            inicios_posibles, fines_posibles, windows_posibles, distancias_posibles):

        if t_ini >= (t_fin - 10):
            continue

        mascara = (tiempo_s >= t_ini) & (tiempo_s <= t_fin)
        t_roi   = tiempo_s[mascara]
        x_roi   = x_relativa[mascara]

        if len(x_roi) < w_len:
            continue

        x_roi_filtrada = savgol_filter(x_roi, window_length=w_len, polyorder=3)
        p_max, _       = find_peaks(x_roi_filtrada, distance=d_min)
        n_pasos        = len(p_max) - 1

        if n_pasos < MIN_PASOS_GLOBAL:
            continue

        periodos          = []
        amplitudes_medias = []
        amplitudes_pp     = []
        minimos           = []   # ← índices de mínimos (en t_roi)

        for i in range(n_pasos):
            idx_act = p_max[i]
            idx_sig = p_max[i + 1]

            T        = t_roi[idx_sig] - t_roi[idx_act]
            segmento = x_roi[idx_act:idx_sig]

            # Mínimo dentro del segmento entre dos picos consecutivos
            idx_min_local  = np.argmin(segmento)
            idx_min_global = idx_act + idx_min_local   # índice en t_roi

            A_pp = x_roi[idx_act] - np.min(segmento)

            periodos.append(T)
            amplitudes_medias.append(A_pp / 2.0)
            amplitudes_pp.append(A_pp)
            minimos.append(idx_min_global)             # ← guardar

        X       = np.array(periodos).reshape(-1, 1)
        y_media = np.array(amplitudes_medias)
        y_pp    = np.array(amplitudes_pp)

        if len(y_media) > 0 and np.max(y_media) > 0:
            y_media = y_media / np.max(y_media)

        if len(y_pp) > 0 and np.max(y_pp) > 0:
            y_pp = y_pp / np.max(y_pp)

        modelo_m = LinearRegression().fit(X, y_media)
        modelo_p = LinearRegression().fit(X, y_pp)

        resultados.append({
            'r2':         modelo_m.score(X, y_media),
            't_inicio':   t_ini,
            't_fin':      t_fin,
            'window_len': w_len,
            'distancia':  d_min,
            'pasos':      n_pasos,
            'X':          X,
            'y_media':    y_media,
            'y_pp':       y_pp,
            'modelo_m':   modelo_m,
            'modelo_p':   modelo_p,
            'minimos':    minimos,   # ← incluido en resultados
        })

    # Seleccionar el mejor por cada umbral
    mejores = {}
    for umbral in UMBRALES:
        candidatos   = [r for r in resultados if r['pasos'] >= umbral]
        mejores[umbral] = max(candidatos, key=lambda x: x['r2']) if candidatos else None

    # Generar gráficas y extraer datos
    output_dir = Path("Gráficas") / exp_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generando figuras y extrayendo datos para {exp_name}...")
    for umbral in UMBRALES:
        m = mejores[umbral]
        if m is None:
            print(f"  MIN={umbral} → sin resultados válidos.")
            continue
        graficar_cuadrado(m, umbral, output_dir, exp_name, tiempo_s, x_relativa)

print("\n✅ PROCESO COMPLETADO. Revisa la consola para los datos numéricos y 'Gráficas/' para los PDFs.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path
from matplotlib.ticker import MaxNLocator


BASE_DIR = Path(r"kinematics")

EXPERIMENTOS = {
    'Ensayo 1': {
        'Invariante':    'invarianeditexp1',
        'Variante':      'varianteditexp1',
        'PotencialSolo': 'potecialsoloeditexp1',
        'Corto':         'cortoeditexp1',
    },
    'Ensayo 2': {
        'Invariante':    'invarianeditexp2',
        'Variante':      'varianteditexp2',
        'PotencialSolo': 'potencialsoloexp2',
        'Corto':         'cortoeditexp2',
    },
    'Ensayo 3': {
        'Invariante':    'invarianeditexp3',
        'Variante':      'varianteeditexp3',
        'PotencialSolo': 'potentialsoloexp3',
        'Corto':         'cortoeditexp3',
    },
}

NOMBRES_LEYENDA = {
    'Invariante':    'Intervalo invariante largo con hiperpolarización',       
    'Variante':      'Intervalo no invariante',         
    'PotencialSolo': 'Intervalo invariante solo con potenciales de acción',          
    'Corto':         'Intervalo invariante corto con hiperpolarización'        
}

COLORES = {
    'Invariante':    'tab:blue',
    'Variante':      'tab:orange',
    'PotencialSolo': 'tab:green',
    'Corto':         'tab:red',
}

n           = 60          # FPS
start_video = 180
end_video   = -60
cm_per_px   = 0.06289
dist_inicio = 0.0
dist_fin    = 93.0        # cm — ventana de análisis ESTRICTA para cálculos

output_dir = Path("Gráficas")
output_dir.mkdir(parents=True, exist_ok=True)


def compute_speed_single(filepath, n, cm_per_px, start_video=0, end_video=-1):
    filepath = Path(filepath)
    if not filepath.exists():
        print(f"  [!] Archivo no encontrado: {filepath.name}")
        return None, None, None

    with open(filepath) as f:
        primera = f.readline()

    tiene_cabecera = not primera.split()[0].replace('.', '').replace('-', '').lstrip('-').isdigit()
    sep    = '\t' if '\t' in primera else ' '
    header = 0 if tiene_cabecera else None

    data  = pd.read_csv(filepath, delimiter=sep, header=header).values
    t_col = data[:, 0]
    factor_tiempo = 1000.0 if t_col.max() > 1000 else 1.0

    segmento = data[start_video:end_video]
    if len(segmento) < n * 4:
        print(f"  [!] Segmento demasiado corto en {filepath.name}")
        return None, None, None

    time_raw = segmento[:, 0] / factor_tiempo
    s_raw    = segmento[:, 3].astype(float)

    if tiene_cabecera:
        col_names = primera.strip().split(sep)
        col3_name = col_names[3] if len(col_names) > 3 else ''
        ya_en_cm  = 'cm' in col3_name
    else:
        ya_en_cm = False

    if not ya_en_cm:
        s_raw = s_raw * cm_per_px

    window_size = n * 2
    kernel = np.ones(window_size) / window_size
    s      = np.convolve(s_raw, kernel, mode='valid')
    time_s = time_raw[window_size - 1:]

    dt = np.diff(time_s)
    dt[dt <= 0] = 1.0 / n
    ds = np.diff(s) / dt
    ds = np.convolve(ds, kernel, mode='valid')

    n_ds     = len(ds)
    time_out = time_s[window_size - 1: window_size - 1 + n_ds]
    s_out    = s_raw[window_size - 1: window_size - 1 + n_ds]
    distance = s_out - s_out[0]

    min_len  = min(len(ds), len(time_out))
    ds       = ds[:min_len]
    distance = distance[:min_len]
    time_out = time_out[:min_len]

    return ds, distance, time_out



fig_dist, axes_dist = plt.subplots(3, 1, figsize=(18, 7), sharex=True, sharey=False)
fig_full, axes_full = plt.subplots(3, 1, figsize=(18, 7), sharex=True, sharey=False)
fig_phase, axes_phase = plt.subplots(3, 1, figsize=(18, 8), sharex=False, sharey=False)
fig_dist.subplots_adjust(hspace=0.18)
fig_full.subplots_adjust(hspace=0.18)
fig_phase.subplots_adjust(hspace=0.22)

for row_idx, (exp, categorias) in enumerate(EXPERIMENTOS.items()):
    ax_dist = axes_dist[row_idx]
    ax_full = axes_full[row_idx]
    ax_phase = axes_phase[row_idx]

    ax_dist.set_title(exp.upper(), fontsize=13, fontweight='bold')
    ax_full.set_title(exp.upper(), fontsize=13, fontweight='bold')
    ax_phase.set_title(f"Retrato de Fase - {exp.upper()}", fontsize=13, fontweight='bold')

    print(f"\n{'='*50}")
    print(f" {exp.upper()} - MÉTRICAS EN TRAMO [{dist_inicio} - {dist_fin} cm]")
    print(f"{'='*50}")

    for categoria, nombre_base in categorias.items():
        filepath = BASE_DIR / f"{nombre_base}_cinematica_completa_cm.txt"
        color    = COLORES[categoria]

        speeds, distances, times = compute_speed_single(filepath, n, cm_per_px, start_video, end_video)

        if speeds is None:
            continue

        # --- APLICAR MÁSCARA ANTES DE CALCULAR ---
        mask = (distances >= dist_inicio) & (distances <= dist_fin)
        
        if not mask.any():
            print(f"  [!] No hay datos en la ventana analítica para {categoria}")
            continue

        # Extraemos solo los datos de la región de interés (ROI)
        roi_speeds = speeds[mask]
        roi_times = times[mask]
        roi_distances = distances[mask]

        # --- 1. CÁLCULO DEL CV (%) EN EL TRAMO ---
        mean_speed = np.mean(roi_speeds)
        std_val = np.std(roi_speeds)
        cv_val = (std_val / mean_speed) * 100 if mean_speed != 0 else 0

        # --- 2. CÁLCULO DEL JERK EN EL TRAMO ---
        dt_array = np.diff(roi_times)
        dt_array[dt_array <= 0] = 1.0 / n
        accel = np.diff(roi_speeds) / dt_array
        
        jerk = np.diff(accel) / dt_array[1:]
        jerk_std = np.std(jerk)

        nombre_mostrar = NOMBRES_LEYENDA.get(categoria, categoria) 
        
        # Imprimir en consola (Estos son los datos hiperprecisos para el TFG)
        print(f"▶ {nombre_mostrar}:")
        print(f"   - Velocidad Media: {mean_speed:.2f} cm/s")
        print(f"   - Desviación (SD): {std_val:.2f} cm/s")
        print(f"   - CV (Variación):  {cv_val:.1f}%")
        print(f"   - Jerk (SD):       {jerk_std:.2f} cm/s³")
        print("-" * 50)

        # Gráfica Completa (Se ve toda la línea, pero la leyenda muestra el CV de la zona útil)
        label_full = f"{nombre_mostrar} (CV={cv_val:.1f}%)"
        ax_full.plot(distances, speeds, color=color, linewidth=1.5, label=label_full, alpha=0.8)

        # Gráfica Ventana Útil
        ax_dist.plot(roi_distances, roi_speeds, color=color, linewidth=1.5, label=label_full, alpha=0.8)
            
        # --- 3. GRÁFICA DE RETRATO DE FASE EN EL TRAMO ---
        min_len_fase = min(len(accel), len(roi_speeds))
        accel_fase = accel[:min_len_fase]
        speeds_fase = roi_speeds[:min_len_fase]
        
        ax_phase.plot(accel_fase, speeds_fase, color=color, linewidth=0.8, alpha=0.7, label=nombre_mostrar)

    # Formato Subplots Velocidad
    for ax in [ax_dist, ax_full]:
        ax.set_ylabel('Velocidad (cm/s)', fontsize=11)
        ax.set_ylim(bottom=0) 
        ymin, ymax = ax.get_ylim() 
        ax.set_ylim(0, ymax * 1.15) 
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, linestyle=':', alpha=0.6)

    # Formato Subplots Fase
    ax_phase.set_ylabel('Velocidad (cm/s)', fontsize=11)
    ax_phase.set_xlabel('Aceleración (cm/s²)', fontsize=11)
    ax_phase.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax_phase.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax_phase.grid(True, linestyle=':', alpha=0.4)

axes_dist[-1].set_xlabel('Distancia (cm)', fontsize=12, fontweight='bold')
axes_full[-1].set_xlabel('Distancia (cm)', fontsize=12, fontweight='bold')

fig_dist.tight_layout()
fig_full.tight_layout()
fig_phase.tight_layout()

out_dist = output_dir / "velocidad_vs_distancia_util_todos_exp.pdf"
out_full = output_dir / "velocidad_vs_distancia_completa_todos_exp.pdf"
out_phase = output_dir / "retrato_fase_aceleracion_vs_velocidad.pdf"

fig_dist.savefig(out_dist, format='pdf', dpi=300, bbox_inches='tight')
fig_full.savefig(out_full, format='pdf', dpi=300, bbox_inches='tight')
fig_phase.savefig(out_phase, format='pdf', dpi=300, bbox_inches='tight')

print(f"\n✅ Guardado:")
print(f"   {out_dist}")
print(f"   {out_full}")
print(f"   {out_phase}")

In [ ]:
import matplotlib.pyplot as plt

# Definir los colores pastel para cada tipo de intervalo (coincidentes con la gráfica)
COLORES_INTERVALO = {
    "Invariante largo (con hiperpolarización)": "#C1DDF6",  # Azul pastel
    "No invariante (Variante)":                 "#FDE4D0",  # Naranja pastel
    "Invariante (solo potenciales de acción)":  "#D6F1C6",  # Verde pastel
    "Invariante corto (con hiperpolarización)": "#F9D9D9"   # Rojo pastel
}

# Datos completos de la tabla (sin la columna de la desviación)
datos = [
    ["Ensayo 1", "Invariante largo (con hiperpolarización)", "1.63 cm/s", "22.3%"],
    ["", "No invariante (Variante)", "1.62 cm/s", "33.2%"],
    ["", "Invariante (solo potenciales de acción)", "1.70 cm/s", "16.0%"],
    ["", "Invariante corto (con hiperpolarización)", "2.12 cm/s", "13.9%"],
    
    ["Ensayo 2", "Invariante largo (con hiperpolarización)", "1.76 cm/s", "20.8%"],
    ["", "No invariante (Variante)", "1.83 cm/s", "23.8%"],
    ["", "Invariante (solo potenciales de acción)", "1.69 cm/s", "18.3%"],
    ["", "Invariante corto (con hiperpolarización)", "2.03 cm/s", "12.8%"],
    
    ["Ensayo 3", "Invariante largo (con hiperpolarización)", "1.64 cm/s", "15.8%"],
    ["", "No invariante (Variante)", "1.86 cm/s", "22.1%"],
    ["", "Invariante (solo potenciales de acción)", "1.68 cm/s", "19.8%"],
    ["", "Invariante corto (con hiperpolarización)", "1.79 cm/s", "17.8%"]
]

columnas = ["Ensayo", "Intervalo de Control", "Velocidad Media", "Variación (CV)"]

# Generar la matriz de colores (ahora con 4 elementos por fila)
colores_filas = []
for fila in datos:
    tipo_intervalo = fila[1]
    color = COLORES_INTERVALO[tipo_intervalo]
    colores_filas.append([color] * 4)

# Configurar la figura (ancho aumentado a 12 para dar espacio a los cm/s)
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('tight')
ax.axis('off')

# Crear la tabla
tabla = ax.table(cellText=datos, 
                 colLabels=columnas, 
                 cellLoc='center', 
                 loc='center',
                 cellColours=colores_filas)

# Ajustes de fuente y escala para presentación académica
tabla.auto_set_font_size(False)
tabla.set_fontsize(10.5)
tabla.scale(1.1, 1.7)

# =============================================================================
# MODIFICACIÓN DE ANCHOS DE COLUMNA PERSONALIZADOS (Reajustado para 4 columnas)
# =============================================================================
# 0: Ensayo, 1: Intervalo, 2: Vel, 3: CV
anchos_columnas = {0: 0.10, 1: 0.44, 2: 0.23, 3: 0.23}

for (fila_idx, col_idx), celda in tabla.get_celld().items():
    if col_idx in anchos_columnas:
        celda.set_width(anchos_columnas[col_idx])
# =============================================================================

# Cabecera en gris oscuro elegante y texto blanco
for j in range(len(columnas)):
    celda_cabecera = tabla[0, j]
    celda_cabecera.set_facecolor('#333333') 
    celda_cabecera.set_text_props(weight='bold', color='white')

# Destacar en negrita las celdas clave (Ensayos y Resultados finales de CV)
for i in range(1, len(datos) + 1):
    tabla[i, 0].set_text_props(weight='bold') # Columna Ensayo
    tabla[i, 3].set_text_props(weight='bold') # Columna CV (ahora es el índice 3)

plt.title("Métricas de estabilidad cinemática en el tramo útil de locomoción", fontweight="bold", pad=20, fontsize=13)
plt.tight_layout()

# Guardar archivos
plt.savefig("Tabla_Estabilidad_Completa.png", dpi=300, bbox_inches='tight')
plt.savefig("Tabla_Estabilidad_Completa.pdf", dpi=300, bbox_inches='tight')
print("¡Hecho! Tabla optimizada (sin SD) guardada en 'Tabla_Estabilidad_Completa.png' y PDF.")